# Microbenchmarks on CPU
This is a notebook for microbenchmarks running on CPU.

In [1]:
# Bootstrap defaults similar to GPU notebook
import os, glob, psutil

# Ensure SPARK_HOME is set (fallback to local build dist)
SPARK_HOME = os.environ.get("SPARK_HOME", "/home/chenqh23/spark/dist")
os.environ["SPARK_HOME"] = SPARK_HOME
os.environ['JAVA_HOME'] = os.environ.get('JAVA_HOME', '/usr/lib/jvm/java-17-openjdk-amd64')

# Clean up lingering SparkSubmit JVMs that can cause empty Py4J answers
for p in psutil.process_iter(['pid','name','cmdline']):
    cl = ' '.join(p.info.get('cmdline') or [])
    if p.info.get('name') == 'java' and 'org.apache.spark.deploy.SparkSubmit' in cl:
        try:
            p.kill()
        except Exception:
            pass

# Ensure findspark can locate Spark
try:
    import findspark; findspark.init(os.environ['SPARK_HOME'])
except Exception:
    pass

# Use local mode unless a cluster URL is provided
os.environ.setdefault("SPARK_MASTER_URL", "local[*]")

# Data and event log defaults
os.environ.setdefault("DATA_ROOT", "/home/chenqh23/spark-rapids-examples/datasets")
os.environ.setdefault("EVENTLOG_DIR", "/tmp/spark-events")
try:
    os.makedirs(os.environ["EVENTLOG_DIR"], exist_ok=True)
except Exception:
    pass

print("SPARK_HOME =", os.environ["SPARK_HOME"]) 
print("SPARK_MASTER_URL =", os.environ["SPARK_MASTER_URL"]) 
print("EVENTLOG_DIR =", os.environ["EVENTLOG_DIR"]) 


SPARK_HOME = /home/chenqh23/spark/dist
SPARK_MASTER_URL = local[*]
EVENTLOG_DIR = /tmp/spark-events


Run the microbenchmark with retry times

In [2]:
def runMicroBenchmark(spark, appName, query, retryTimes):
    count = 0
    total_time = 0
    # You can print the physical plan of each query
    # spark.sql(query).explain()
    while count < retryTimes:
        start = time.time()
        spark.sql(query).collect()
        end = time.time()
        total_time += round(end - start, 2)
        count = count + 1
        print("Retry times : {}, ".format(count) + appName + " microbenchmark takes {} seconds".format(round(end - start, 2)))
    print(appName + " microbenchmark takes average {} seconds after {} retries".format(round(total_time/retryTimes),retryTimes))
    with open('result.txt', 'a') as file:
        file.write("{},{},{}\n".format(appName, round(total_time/retryTimes), retryTimes))

In [ ]:
# You need to update data path with your real path and hardware resource!
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import time, os

# Java 17 add-opens flags (harmless on Java 8+)
_DEF_OPENS = (
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED "
    "--add-opens=java.base/jdk.internal.ref=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED "
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED "
    "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED"
)

# Build base conf (GPU-style settings but CPU-only)
base = (SparkConf()
    .setMaster(os.environ.get("SPARK_MASTER_URL", "local[*]"))
    .setAppName("Microbenchmark on CPU")
    .set("spark.driver.memory", os.environ.get("DRIVER_MEM", "12g"))
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.files.maxPartitionBytes", os.environ.get("MAX_PARTITION_BYTES", "128m"))
    .set("spark.sql.shuffle.partitions", os.environ.get("SHUFFLE_PARTITIONS", "96"))
    .set("spark.locality.wait", "0")
    .set("spark.scheduler.mode", "FAIR")
    .set("spark.eventLog.enabled", "false")
    .set("spark.driver.extraJavaOptions", _DEF_OPENS)
    .set("spark.executor.extraJavaOptions", _DEF_OPENS)
    # IAA compression codec wiring
    .set("spark.jars", "/home/chenqh23/offload/iaa-compress/iaa-codec.jar")
    .set("spark.io.compression.codec", "org.apache.spark.io.IaaCompressionCodec")
    .set("spark.io.compression.iaa.blockSize", os.environ.get("IAA_BLOCK_SIZE", "262144"))
    .set("spark.driver.extraLibraryPath", "/home/chenqh23/offload/iaa-compress/native/build:/home/chenqh23/qpl_install_dir/lib")
    .set("spark.executor.extraLibraryPath", "/home/chenqh23/offload/iaa-compress/native/build:/home/chenqh23/qpl_install_dir/lib")
)

# create spark session
spark = SparkSession.builder.config(conf=base).getOrCreate()


25/11/19 14:26:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
# IAA sanity check: confirm hardware path availability via QPL HW job init
jvm = spark._jvm
iaa = jvm.org.apache.spark.io.IAA
try:
    hw_ok = bool(iaa.isHardwareAvailable())
except Exception as e:
    hw_ok = False
    print("IAA isHardwareAvailable() check failed:", e)
print("IAA hardware available:", hw_ok)

# Note: codec in use
print("spark.io.compression.codec =", spark.conf.get("spark.io.compression.codec"))


IAA hardware available: False
spark.io.compression.codec = org.apache.spark.io.IaaCompressionCodec


In [5]:
# Helpers to capture IAA compression metrics per-SQL and print deltas
from typing import Tuple, List

def get_iaa_metrics():
    m = spark._jvm.org.apache.spark.io.IaaCompressionMetrics
    bounds_kb = list(m.getBucketBoundsKB())
    # Use uncompressed-size buckets by default
    counts = list(m.getUncompressedBucketCounts())
    totals = list(m.getTotals())  # [blocks, uncomp_bytes, comp_bytes]
    return bounds_kb, counts, totals


def print_iaa_hist_delta(title: str, before, after):
    bounds_kb_b, counts_b, totals_b = before
    bounds_kb_a, counts_a, totals_a = after
    # assume bounds unchanged
    counts_delta = [a - b for a, b in zip(counts_a, counts_b)]
    blocks = totals_a[0] - totals_b[0]
    uncomp = totals_a[1] - totals_b[1]
    comp = totals_a[2] - totals_b[2]

    # Build labels
    labels = []
    start = 0
    for b in bounds_kb_b:
        if b >= 2**30:  # sentinel for Int.MaxValue
            labels.append(f">={start}KB")
        else:
            labels.append(f"{start}-{b}KB")
        start = b

    print(f"IAA uncompressed-size histogram for {title} (this run only):")
    print("Total blocks:", blocks)
    if blocks > 0:
        print("Total uncompressed MB:", round(uncomp/1024/1024, 2))
        print("Total compressed MB:", round(comp/1024/1024, 2))
    for lbl, cnt in zip(labels, counts_delta):
        if cnt:
            print(f"{lbl}: {cnt}")



In [6]:
dataRoot = os.environ.get("DATA_ROOT", "/home/chenqh23/spark-rapids-examples/datasets")

# Load dataframe and create tempView (avoid extreme repartition)
spark.read.parquet(dataRoot + "/tpcds/customer").createOrReplaceTempView("customer")
spark.read.parquet(dataRoot + "/tpcds/store_sales").createOrReplaceTempView("store_sales")
spark.read.parquet(dataRoot + "/tpcds/catalog_sales").createOrReplaceTempView("catalog_sales")
spark.read.parquet(dataRoot + "/tpcds/web_sales").createOrReplaceTempView("web_sales")
spark.read.parquet(dataRoot + "/tpcds/item").createOrReplaceTempView("item")
spark.read.parquet(dataRoot + "/tpcds/date_dim").createOrReplaceTempView("date_dim")

# Cache hot tables to reduce I/O contention during parallel runs
for t in ("customer","store_sales","catalog_sales","web_sales","item","date_dim"):
    try:
        spark.catalog.cacheTable(t)
    except Exception:
        pass
# Materialize caches once to warm up
for t in ("customer","store_sales","catalog_sales","web_sales","item","date_dim"):
    _ = spark.table(t).count()

print("-"*50)
time.sleep(2)

25/11/19 14:26:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


--------------------------------------------------


### Expand&HashAggregate
This is a microbenchmark about Expand&HashAggregate expressions running on the CPU. The query calculates the distinct value of some dimension columns and average birth year by different c_salutation of customers after grouping by c_current_hdemo_sk.

In [7]:
query0 = '''
select c_current_hdemo_sk,
count(DISTINCT if(c_salutation=="Ms.",c_salutation,null)) as c1,
count(DISTINCT if(c_salutation=="Mr.",c_salutation,null)) as c12,
count(DISTINCT if(c_salutation=="Dr.",c_salutation,null)) as c13,

count(DISTINCT if(c_salutation=="Ms.",c_first_name,null)) as c2,
count(DISTINCT if(c_salutation=="Mr.",c_first_name,null)) as c22,
count(DISTINCT if(c_salutation=="Dr.",c_first_name,null)) as c23,

count(DISTINCT if(c_salutation=="Ms.",c_last_name,null)) as c3,
count(DISTINCT if(c_salutation=="Mr.",c_last_name,null)) as c32,
count(DISTINCT if(c_salutation=="Dr.",c_last_name,null)) as c33,

count(DISTINCT if(c_salutation=="Ms.",c_birth_country,null)) as c4,
count(DISTINCT if(c_salutation=="Mr.",c_birth_country,null)) as c42,
count(DISTINCT if(c_salutation=="Dr.",c_birth_country,null)) as c43,

count(DISTINCT if(c_salutation=="Ms.",c_email_address,null)) as c5,
count(DISTINCT if(c_salutation=="Mr.",c_email_address,null)) as c52,
count(DISTINCT if(c_salutation=="Dr.",c_email_address,null)) as c53,

count(DISTINCT if(c_salutation=="Ms.",c_login,null)) as c6,
count(DISTINCT if(c_salutation=="Mr.",c_login,null)) as c62,
count(DISTINCT if(c_salutation=="Dr.",c_login,null)) as c63,

count(DISTINCT if(c_salutation=="Ms.",c_preferred_cust_flag,null)) as c7,
count(DISTINCT if(c_salutation=="Mr.",c_preferred_cust_flag,null)) as c72,
count(DISTINCT if(c_salutation=="Dr.",c_preferred_cust_flag,null)) as c73,

count(DISTINCT if(c_salutation=="Ms.",c_birth_month,null)) as c8,
count(DISTINCT if(c_salutation=="Mr.",c_birth_month,null)) as c82,
count(DISTINCT if(c_salutation=="Dr.",c_birth_month,null)) as c83,

avg(if(c_salutation=="Ms.",c_birth_year,null)) as avg1,
avg(if(c_salutation=="Mr.",c_birth_year,null)) as avg2,
avg(if(c_salutation=="Dr.",c_birth_year,null)) as avg3,
avg(if(c_salutation=="Miss.",c_birth_year,null)) as avg4,
avg(if(c_salutation=="Mrs.",c_birth_year,null)) as avg5,
avg(if(c_salutation=="Sir.",c_birth_year,null)) as avg6,
avg(if(c_salutation=="Professor.",c_birth_year,null)) as avg7,
avg(if(c_salutation=="Teacher.",c_birth_year,null)) as avg8,
avg(if(c_salutation=="Agent.",c_birth_year,null)) as avg9,
avg(if(c_salutation=="Director.",c_birth_year,null)) as avg10
from customer group by c_current_hdemo_sk
'''
print("-"*50)

--------------------------------------------------


In [8]:
# Run microbenchmark with n retry time
before = get_iaa_metrics()
runMicroBenchmark(spark,"Expand&HashAggregate", query0, 2)
after = get_iaa_metrics()
print_iaa_hist_delta("Expand&HashAggregate", before, after)

time.sleep(2)

Retry times : 1, Expand&HashAggregate microbenchmark takes 11.04 seconds


Retry times : 2, Expand&HashAggregate microbenchmark takes 10.1 seconds
Expand&HashAggregate microbenchmark takes average 11 seconds after 2 retries
IAA uncompressed-size histogram for Expand&HashAggregate (this run only):
Total blocks: 187938
Total uncompressed MB: 1200.53
Total compressed MB: 95.26
0-0KB: 28832
0-1KB: 1540
1-2KB: 34
2-4KB: 40
4-8KB: 157492


### Windowing (without data skew)
This is a microbenchmark about windowing expressions running on CPU mode. The sub-query calculates the average ss_sales_price of a fixed window function partition by ss_customer_sk, and the parent query calculates the average price of the sub-query grouping by each customer.

In [9]:
query1 = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
where ss_customer_sk is not null
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)

--------------------------------------------------


In [10]:
# Run microbenchmark with n retry time
before = get_iaa_metrics()
runMicroBenchmark(spark,"Windowing without skew", query1, 2)
after = get_iaa_metrics()
print_iaa_hist_delta("Windowing without skew", before, after)

time.sleep(2)

Retry times : 1, Windowing without skew microbenchmark takes 32.14 seconds


Retry times : 2, Windowing without skew microbenchmark takes 11.29 seconds
Windowing without skew microbenchmark takes average 22 seconds after 2 retries
IAA uncompressed-size histogram for Windowing without skew (this run only):
Total blocks: 3516916
Total uncompressed MB: 2998.63
Total compressed MB: 853.96
0-0KB: 2188395
0-1KB: 1081081
1-2KB: 612
2-4KB: 1117
4-8KB: 245711


### Windowing(with data skew)
Data skew is caused by many null values in the ss_customer_sk column.

In [11]:
query2 = '''
select ss_customer_sk,avg(avg_price) as avg_price
from
(
SELECT ss_customer_sk ,avg(ss_sales_price) OVER (PARTITION BY ss_customer_sk order by ss_sold_date_sk ROWS BETWEEN 50 PRECEDING AND 50 FOLLOWING ) as avg_price
FROM store_sales
) group by ss_customer_sk order by 2 desc 
'''
print("-"*50)

--------------------------------------------------


In [12]:
# Run microbenchmark with n retry time
before = get_iaa_metrics()
runMicroBenchmark(spark,"Windowing with skew", query2, 2)
after = get_iaa_metrics()
print_iaa_hist_delta("Windowing with skew", before, after)

time.sleep(2)

Retry times : 1, Windowing with skew microbenchmark takes 55.43 seconds


Retry times : 2, Windowing with skew microbenchmark takes 56.05 seconds
Windowing with skew microbenchmark takes average 56 seconds after 2 retries
IAA uncompressed-size histogram for Windowing with skew (this run only):
Total blocks: 13500791
Total uncompressed MB: 2559.75
Total compressed MB: 722.02
0-0KB: 13180474
0-1KB: 61482
1-2KB: 590
2-4KB: 1099
4-8KB: 257146


### Intersection
This is a microbenchmark about intersection operation running on CPU mode. The query calculates items in the same brand, class, and category that are sold in all three sales channels in two consecutive years.

In [13]:
query3 = '''
select i_item_sk ss_item_sk
 from item,
    (select iss.i_brand_id brand_id, iss.i_class_id class_id, iss.i_category_id category_id
     from store_sales, item iss, date_dim d1
     where ss_item_sk = iss.i_item_sk
                    and ss_sold_date_sk = d1.d_date_sk
       and d1.d_year between 1999 AND 1999 + 2
   intersect
     select ics.i_brand_id, ics.i_class_id, ics.i_category_id
     from catalog_sales, item ics, date_dim d2
     where cs_item_sk = ics.i_item_sk
       and cs_sold_date_sk = d2.d_date_sk
       and d2.d_year between 1999 AND 1999 + 2
   intersect
     select iws.i_brand_id, iws.i_class_id, iws.i_category_id
     from web_sales, item iws, date_dim d3
     where ws_item_sk = iws.i_item_sk
       and ws_sold_date_sk = d3.d_date_sk
       and d3.d_year between 1999 AND 1999 + 2) x
 where i_brand_id = brand_id
   and i_class_id = class_id
   and i_category_id = category_id
'''

In [14]:
# Run microbenchmark with n retry time
before = get_iaa_metrics()
runMicroBenchmark(spark,"NDS Q14a subquery", query3, 2)
after = get_iaa_metrics()
print_iaa_hist_delta("NDS Q14a subquery", before, after)

time.sleep(2)

Retry times : 1, NDS Q14a subquery microbenchmark takes 5.31 seconds


Retry times : 2, NDS Q14a subquery microbenchmark takes 4.02 seconds
NDS Q14a subquery microbenchmark takes average 5 seconds after 2 retries
IAA uncompressed-size histogram for NDS Q14a subquery (this run only):
Total blocks: 2208716
Total uncompressed MB: 1054.57
Total compressed MB: 180.26
0-0KB: 2028260
0-1KB: 51778
1-2KB: 1524
2-4KB: 3038
4-8KB: 124116


In [15]:
# Uncompressed size distribution (IAA codec)
# Buckets are upper-bounds in KB: counts reflect uncompressed block sizes per bucket
jvm = spark._jvm
metrics = jvm.org.apache.spark.io.IaaCompressionMetrics
bounds_kb = list(metrics.getBucketBoundsKB())
counts = list(metrics.getUncompressedBucketCounts())
blocks, uncomp_bytes, comp_bytes = list(metrics.getTotals())

# Build human-readable bucket labels
labels = []
start = 0
for b in bounds_kb:
    if b >= 2**30:  # Int.MaxValue sentinel
        labels.append(f">={start}KB")
    else:
        labels.append(f"{start}-{b}KB")
    start = b

print("Total blocks:", blocks)
print("Total uncompressed MB:", round(uncomp_bytes/1024/1024, 2))
print("Total compressed MB:", round(comp_bytes/1024/1024, 2))

print("\nUncompressed size histogram (KB):")
for lbl, cnt in zip(labels, counts):
    if cnt:
        print(f"{lbl}: {cnt}")


Total blocks: 19417568
Total uncompressed MB: 7815.52
Total compressed MB: 1852.45

Uncompressed size histogram (KB):
0-0KB: 17427093
0-1KB: 1197956
1-2KB: 2760
2-4KB: 5294
4-8KB: 784465


In [16]:
# Run the 4 micro-benchmarks concurrently on one SparkSession/CPU
# Requires: query0, query1, query2, query3 already defined; temp views already created.

from concurrent.futures import ThreadPoolExecutor, as_completed
import time, os

# Tune quickly if you hit contention
spark.conf.set("spark.sql.files.maxPartitionBytes", os.environ.get("MB_MAX_PART_BYTES", "128m"))

def run_queries_in_pool(pool_name: str, query, retryTimes: int = 1):
    count = 0
    sc = spark.sparkContext
    t0 = time.time()
    sc.setLocalProperty("spark.scheduler.pool", pool_name)  # assign this job to a pool
    sc.setJobGroup(f"{pool_name}", f"{pool_name} run", True)
    while count < retryTimes:
        print(f"run {count+1} of {retryTimes} for {pool_name}")
        spark.sql(query).collect()  # blocking action submits a job
        count += 1
    return pool_name, round(time.time() - t0, 1)

jobs = [
    ("poolA", query0),
    ("poolB", query1),
    ("poolC", query2),
    ("poolD", query3),
    ("poolE", query0),
    ("poolF", query1),
    ("poolG", query2),
    ("poolH", query3),
    
]

# You can tune the parallelism here quickly if you hit contention
PARALLEL_JOBS = int(os.environ.get("MB_PARALLEL_JOBS", "8"))

with ThreadPoolExecutor(max_workers=PARALLEL_JOBS) as ex:
    futs = [ex.submit(run_queries_in_pool, p, query) for p, query in jobs[:PARALLEL_JOBS]]
    for f in as_completed(futs):
        name, secs = f.result()
        print(f"{name} took {secs}s")



run 1 of 1 for poolA
run 1 of 1 for poolB
run 1 of 1 for poolC
run 1 of 1 for poolD
run 1 of 1 for poolE
run 1 of 1 for poolF
run 1 of 1 for poolG
run 1 of 1 for poolH


25/11/19 14:30:10 WARN FairSchedulableBuilder: A job was submitted with scheduler pool poolG, which has not been configured. This can happen when the file that pools are read from isn't set, or when that file doesn't contain poolG. Created poolG with default configuration (schedulingMode: FIFO, minShare: 0, weight: 1)
25/11/19 14:30:10 WARN FairSchedulableBuilder: A job was submitted with scheduler pool poolC, which has not been configured. This can happen when the file that pools are read from isn't set, or when that file doesn't contain poolC. Created poolC with default configuration (schedulingMode: FIFO, minShare: 0, weight: 1)
25/11/19 14:30:10 WARN FairSchedulableBuilder: A job was submitted with scheduler pool poolF, which has not been configured. This can happen when the file that pools are read from isn't set, or when that file doesn't contain poolF. Created poolF with default configuration (schedulingMode: FIFO, minShare: 0, weight: 1)
25/11/19 14:30:10 WARN FairSchedulableBu

poolD took 91.5s
poolH took 91.6s


poolF took 96.7s
poolB took 97.2s


25/11/19 14:31:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/19 14:31:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/19 14:31:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/19 14:31:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/19 14:31:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/19 14:31:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/19 14:31:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/19 14:31:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/11/19 14:31:50 WARN RowBasedKeyValueBatch: Calling spill() on

poolA took 110.1s


poolE took 111.0s


poolG took 127.9s


poolC took 144.0s
